<a href="https://colab.research.google.com/github/rbuzmaa/Research-Project/blob/main/colon_cancer_cross_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-Dataset Colorectal Cancer Detection
## Train on LC25000 → Evaluate on Colorectal Histology MNIST

### Methodology Overview
This notebook implements a cross-dataset generalization study:
1. **Train** a CNN on LC25000 (all 5 classes, augmented images)
2. **Freeze** trained weights — no retraining on new data
3. **Evaluate** on Colorectal Histology MNIST (2 mappable classes only)
4. **Analyse** generalization performance across domain shift

### Class Mapping
| LC25000 (Train) | Colorectal MNIST (Test) | Rationale |
|---|---|---|
| colon_aca (adenocarcinoma) | tumor | Both = malignant colorectal tissue |
| colon_n (benign) | mucosa | Both = normal colorectal tissue |
| lung_aca, lung_scc, lung_n | — | No equivalent class — excluded from eval |

## Recovery Cells

In [ ]:
!pip install pytorch-grad-cam --quiet

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import shutil
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    accuracy_score, f1_score
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Restore directory structure
BASE_DIR      = Path('/content/cancer_detection')
LC25000_DIR   = BASE_DIR / 'lc25000'
COLORECTAL_DIR= BASE_DIR / 'colorectal'
MODEL_DIR     = BASE_DIR / 'models'
RESULTS_DIR   = BASE_DIR / 'results'

for d in [BASE_DIR, LC25000_DIR, COLORECTAL_DIR, MODEL_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / 'best_lc25000_model.pth'

# Copy model back from Drive if not already present
DRIVE_DIR = Path('/content/drive/MyDrive/cancer_detection_results')

if not BEST_MODEL_PATH.exists():
    shutil.copy(DRIVE_DIR / 'best_lc25000_model.pth', BEST_MODEL_PATH)
    print('Model restored from Drive.')
else:
    print('Model already present.')

print(f'Model path exists: {BEST_MODEL_PATH.exists()}')

In [ ]:
# ── LC25000 class paths ───────────────────────────────────────
LC25000_CLASSES = ['colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc']

LC25000_CLASS_PATHS = {}
for cls in LC25000_CLASSES:
    for p in LC25000_DIR.rglob(cls):
        if p.is_dir():
            LC25000_CLASS_PATHS[cls] = p
            break

print('LC25000 class paths:')
for cls, path in LC25000_CLASS_PATHS.items():
    count = len(list(path.glob('*.jpeg')) +
                list(path.glob('*.jpg')) +
                list(path.glob('*.png'))) if path.exists() else 0
    print(f'  {cls}: {count} images')

# ── Colorectal MNIST class paths ──────────────────────────────
COLORECTAL_FOLDER_MAP = {
    'tumor'  : ['01_TUMOR',  'TUMOR',  'tumor'],
    'mucosa' : ['06_MUCOSA', 'MUCOSA', 'mucosa'],
    'stroma' : ['02_STROMA', 'STROMA', 'stroma'],
    'complex': ['03_COMPLEX','COMPLEX','complex'],
    'lympho' : ['04_LYMPHO', 'LYMPHO', 'lympho'],
    'debris' : ['05_DEBRIS', 'DEBRIS', 'debris'],
    'adipose': ['07_ADIPOSE','ADIPOSE','adipose'],
    'empty'  : ['08_EMPTY',  'EMPTY',  'empty'],
}
COLORECTAL_CLASSES_MAPPED = ['tumor', 'mucosa']

COLORECTAL_CLASS_PATHS = {}
for cls_name, possible_folders in COLORECTAL_FOLDER_MAP.items():
    for folder in possible_folders:
        for p in COLORECTAL_DIR.rglob(folder):
            if p.is_dir():
                COLORECTAL_CLASS_PATHS[cls_name] = p
                break
        if cls_name in COLORECTAL_CLASS_PATHS:
            break

print('\nColorectal MNIST class paths:')
for cls, path in COLORECTAL_CLASS_PATHS.items():
    count = len(list(path.glob('*.tif')) +
                list(path.glob('*.png')) +
                list(path.glob('*.jpg'))) if path.exists() else 0
    used = '← USED' if cls in COLORECTAL_CLASSES_MAPPED else ''
    print(f'  {cls}: {count} images  {used}')

In [ ]:
# Colab disk resets between sessions
# Check if images are still present — if not, redownload

lc_ok = all(
    p.exists() and len(list(p.glob('*.jpeg')) +
                        list(p.glob('*.jpg')) +
                        list(p.glob('*.png'))) > 0
    for p in LC25000_CLASS_PATHS.values()
) if LC25000_CLASS_PATHS else False

cr_ok = all(
    p.exists() and len(list(p.glob('*.tif')) +
                        list(p.glob('*.png')) +
                        list(p.glob('*.jpg'))) > 0
    for cls, p in COLORECTAL_CLASS_PATHS.items()
    if cls in COLORECTAL_CLASSES_MAPPED
) if COLORECTAL_CLASS_PATHS else False

print(f'LC25000 images on disk     : {lc_ok}')
print(f'Colorectal images on disk  : {cr_ok}')

if not lc_ok or not cr_ok:
    print('\nDatasets not found — redownloading...')
    # Upload kaggle.json first
    from google.colab import files
    uploaded = files.upload()
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
    os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

    if not lc_ok:
        print('Downloading LC25000...')
        !kaggle datasets download -d andrewmvd/lung-and-colon-cancer-histopathological-images \
            -p /content/cancer_detection/lc25000 --unzip --quiet
        print('LC25000 done.')

    if not cr_ok:
        print('Downloading Colorectal MNIST...')
        !kaggle datasets download -d kmader/colorectal-histology-mnist \
            -p /content/cancer_detection/colorectal --unzip --quiet
        print('Colorectal MNIST done.')

    # Rebuild paths after download
    LC25000_CLASS_PATHS = {}
    for cls in LC25000_CLASSES:
        for p in LC25000_DIR.rglob(cls):
            if p.is_dir():
                LC25000_CLASS_PATHS[cls] = p
                break

    COLORECTAL_CLASS_PATHS = {}
    for cls_name, possible_folders in COLORECTAL_FOLDER_MAP.items():
        for folder in possible_folders:
            for p in COLORECTAL_DIR.rglob(folder):
                if p.is_dir():
                    COLORECTAL_CLASS_PATHS[cls_name] = p
                    break
            if cls_name in COLORECTAL_CLASS_PATHS:
                break
    print('Paths rebuilt after download.')
else:
    print('All images present — no download needed.')

In [ ]:
import os

UNIFIED_DIR = BASE_DIR / 'lc25000_unified'
UNIFIED_DIR.mkdir(exist_ok=True)

for cls, cls_path in LC25000_CLASS_PATHS.items():
    link_path = UNIFIED_DIR / cls
    if link_path.exists() or link_path.is_symlink():
        link_path.unlink()
    os.symlink(cls_path, link_path)

from torchvision.datasets import ImageFolder

IMG_SIZE       = 224
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Rebuild class index mapping from ImageFolder
temp_dataset        = ImageFolder(root=str(UNIFIED_DIR))
LC25000_CLASS_TO_IDX = temp_dataset.class_to_idx
LC25000_IDX_TO_CLASS = {v: k for k, v in LC25000_CLASS_TO_IDX.items()}
NUM_CLASSES          = len(LC25000_CLASS_TO_IDX)

print('LC25000 class mapping restored:')
for cls, idx in LC25000_CLASS_TO_IDX.items():
    print(f'  {idx}: {cls}')

In [ ]:
def build_model(num_classes):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes)
    )
    return model

model = build_model(NUM_CLASSES).to(device)
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

for param in model.parameters():
    param.requires_grad = False
model.eval()

print(f'Model loaded from epoch {checkpoint["epoch"]}')
print(f'LC25000 val accuracy: {checkpoint["val_acc"]:.4f}')
lc25000_val_acc = checkpoint['val_acc']

In [ ]:
class ColorectalMappedDataset(Dataset):
    def __init__(self, class_paths, lc25000_class_to_idx, transform=None):
        self.transform = transform
        self.samples   = []
        LABEL_MAP = {'tumor': 'colon_aca', 'mucosa': 'colon_n'}
        print('Class mapping:')
        for cr_cls, lc_cls in LABEL_MAP.items():
            lc_idx   = lc25000_class_to_idx[lc_cls]
            cls_path = class_paths.get(cr_cls)
            if cls_path is None or not cls_path.exists():
                print(f'  WARNING: {cr_cls} path not found')
                continue
            img_files = (list(cls_path.glob('*.tif')) +
                         list(cls_path.glob('*.png')) +
                         list(cls_path.glob('*.jpg')))
            for f in img_files:
                self.samples.append((str(f), lc_idx, cr_cls))
            print(f'  {cr_cls} → {lc_cls} (idx={lc_idx}) | {len(img_files)} images')
        print(f'Total: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, _ = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

colorectal_test_dataset = ColorectalMappedDataset(
    class_paths          = COLORECTAL_CLASS_PATHS,
    lc25000_class_to_idx = LC25000_CLASS_TO_IDX,
    transform            = val_test_transforms
)
colorectal_test_loader = DataLoader(
    colorectal_test_dataset, batch_size=32,
    shuffle=False, num_workers=2, pin_memory=True
)

# Rerun inference
all_labels, all_preds, all_probs = [], [], []
with torch.no_grad():
    for images, labels in colorectal_test_loader:
        images  = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)
print(f'Inference complete — {len(all_labels)} images')

# Rebuild restricted evaluation variables
COLON_INDICES = [LC25000_CLASS_TO_IDX['colon_aca'],
                 LC25000_CLASS_TO_IDX['colon_n']]

all_labels_filtered, all_preds_filtered, all_probs_filtered = [], [], []
with torch.no_grad():
    for images, labels in colorectal_test_loader:
        images  = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        colon_probs = probs[:, COLON_INDICES]
        colon_probs = colon_probs / colon_probs.sum(dim=1, keepdim=True)
        _, colon_pred_idx = colon_probs.max(1)
        final_preds = torch.tensor([COLON_INDICES[i]
                                    for i in colon_pred_idx])
        all_labels_filtered.extend(labels.numpy())
        all_preds_filtered.extend(final_preds.numpy())
        all_probs_filtered.extend(colon_probs.cpu().numpy())

all_labels_filtered = np.array(all_labels_filtered)
all_preds_filtered  = np.array(all_preds_filtered)
all_probs_filtered  = np.array(all_probs_filtered)
print(f'Restricted inference complete — {len(all_labels_filtered)} images')
print('All variables ready — you can now run any analysis cell.')

## Threshold Sensitivity Analysis

In [ ]:
# ============================================================
# Threshold Sensitivity Analysis
# Test different decision thresholds on restricted evaluation
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, f1_score,
                             accuracy_score)

# Get cancer probability from restricted evaluation
# all_probs_filtered[:, 0] = colon_aca probability after renormalisation
y_true_binary = (all_labels_filtered ==
                 LC25000_CLASS_TO_IDX['colon_aca']).astype(int)
cancer_probs  = all_probs_filtered[:, 0]

# Test thresholds from 0.1 to 0.9
thresholds   = np.arange(0.1, 0.95, 0.05)
sensitivities = []
specificities = []
f1_scores     = []
accuracies    = []

for thresh in thresholds:
    y_pred_t = (cancer_probs >= thresh).astype(int)
    cm = confusion_matrix(y_true_binary, y_pred_t)

    if cm.shape == (2, 2):
        TP = cm[1, 1]
        FN = cm[1, 0]
        FP = cm[0, 1]
        TN = cm[0, 0]
        sens = TP / (TP + FN) if (TP + FN) > 0 else 0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0
    else:
        sens, spec = 0, 0

    sensitivities.append(sens)
    specificities.append(spec)
    f1_scores.append(f1_score(y_true_binary, y_pred_t,
                              zero_division=0))
    accuracies.append(accuracy_score(y_true_binary, y_pred_t))

# ── Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Threshold Sensitivity Analysis — Restricted Evaluation',
             fontweight='bold', fontsize=13)

# Sensitivity vs Specificity across thresholds
axes[0].plot(thresholds, sensitivities, 'o-',
             color='#C0392B', linewidth=2,
             markersize=6, label='Sensitivity (cancer recall)')
axes[0].plot(thresholds, specificities, 's-',
             color='#1A7A5E', linewidth=2,
             markersize=6, label='Specificity (benign recall)')
axes[0].plot(thresholds, f1_scores, '^-',
             color='#534AB7', linewidth=2,
             markersize=6, label='F1 Score')
axes[0].axvline(x=0.5, color='gray', linestyle='--',
                linewidth=1.2, label='Default threshold (0.5)')

# Find crossover point
cross_idx = np.argmin(np.abs(
    np.array(sensitivities) - np.array(specificities)
))
axes[0].axvline(x=thresholds[cross_idx], color='orange',
                linestyle='--', linewidth=1.2,
                label=f'Crossover ({thresholds[cross_idx]:.2f})')

axes[0].set_xlabel('Decision Threshold', fontweight='bold')
axes[0].set_ylabel('Score', fontweight='bold')
axes[0].set_title('Sensitivity vs Specificity\nacross thresholds',
                  fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_xlim([0.05, 1.0])
axes[0].set_ylim([0, 1.05])

# Accuracy and F1 across thresholds
axes[1].plot(thresholds, accuracies, 'o-',
             color='#4C72B0', linewidth=2,
             markersize=6, label='Accuracy')
axes[1].plot(thresholds, f1_scores, 's-',
             color='#534AB7', linewidth=2,
             markersize=6, label='F1 Score')
axes[1].axvline(x=0.5, color='gray', linestyle='--',
                linewidth=1.2, label='Default threshold (0.5)')

best_f1_idx = np.argmax(f1_scores)
axes[1].axvline(x=thresholds[best_f1_idx], color='green',
                linestyle='--', linewidth=1.2,
                label=f'Best F1 threshold ({thresholds[best_f1_idx]:.2f})')

axes[1].set_xlabel('Decision Threshold', fontweight='bold')
axes[1].set_ylabel('Score', fontweight='bold')
axes[1].set_title('Accuracy & F1\nacross thresholds',
                  fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_xlim([0.05, 1.0])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'threshold_sensitivity_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Print summary table ───────────────────────────────────────
print('Threshold Sensitivity Analysis:')
print(f'{"Threshold":>10} {"Sensitivity":>12} '
      f'{"Specificity":>12} {"F1":>8} {"Accuracy":>10}')
print('-' * 56)
for i, t in enumerate(thresholds):
    print(f'{t:>10.2f} {sensitivities[i]:>12.4f} '
          f'{specificities[i]:>12.4f} '
          f'{f1_scores[i]:>8.4f} {accuracies[i]:>10.4f}')

print(f'\nDefault threshold (0.5):')
default_idx = np.argmin(np.abs(thresholds - 0.5))
print(f'  Sensitivity : {sensitivities[default_idx]:.4f}')
print(f'  Specificity : {specificities[default_idx]:.4f}')

print(f'\nBest F1 threshold  : {thresholds[best_f1_idx]:.2f}')
print(f'  Sensitivity : {sensitivities[best_f1_idx]:.4f}')
print(f'  Specificity : {specificities[best_f1_idx]:.4f}')
print(f'  F1          : {f1_scores[best_f1_idx]:.4f}')

print(f'\nSensitivity/Specificity crossover threshold: '
      f'{thresholds[cross_idx]:.2f}')
print(f'  Sensitivity : {sensitivities[cross_idx]:.4f}')
print(f'  Specificity : {specificities[cross_idx]:.4f}')

## Step 1 — Install & Import Dependencies

In [ ]:
# Install kaggle API for dataset download
!pip install kaggle --quiet

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import zipfile
import shutil
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from PIL import Image

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    f1_score
)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2 — Mount Google Drive & Set Up Kaggle API

Upload your `kaggle.json` API token to Colab before running this cell.  
Get it from: https://www.kaggle.com/settings → API → Create New Token

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

# Upload kaggle.json when prompted
print('Please upload your kaggle.json file:')
uploaded = files.upload()

# Place kaggle credentials
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle API configured successfully.')

## Step 3 — Download Datasets

### Dataset 1: LC25000
LC25000 contains 25,000 histopathology images across 5 classes (5,000 per class):  
- `colon_aca` — colon adenocarcinoma (malignant)  
- `colon_n` — colon normal/benign  
- `lung_aca` — lung adenocarcinoma  
- `lung_scc` — lung squamous cell carcinoma  
- `lung_n` — lung normal/benign  

### Dataset 2: Colorectal Histology MNIST
Contains 5,000 images across 8 tissue classes.  
We will only use: `tumor` and `mucosa`

In [ ]:
# Create working directories
BASE_DIR = Path('/content/cancer_detection')
LC25000_DIR = BASE_DIR / 'lc25000'
COLORECTAL_DIR = BASE_DIR / 'colorectal'
MODEL_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'

for d in [LC25000_DIR, COLORECTAL_DIR, MODEL_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directory structure created:')
for d in [LC25000_DIR, COLORECTAL_DIR, MODEL_DIR, RESULTS_DIR]:
    print(f'  {d}')

In [ ]:
# Download LC25000
# NOTE: Search kaggle for 'LC25000 lung and colon histopathological image dataset'
# Replace the dataset slug below with the correct one from Kaggle
print('Downloading LC25000...')
!kaggle datasets download -d andrewmvd/lung-and-colon-cancer-histopathological-images -p /content/cancer_detection/lc25000 --unzip --quiet
print('LC25000 downloaded.')

# Download Colorectal Histology MNIST
print('Downloading Colorectal Histology MNIST...')
!kaggle datasets download -d kmader/colorectal-histology-mnist -p /content/cancer_detection/colorectal --unzip --quiet
print('Colorectal Histology MNIST downloaded.')

In [ ]:
# Verify downloaded structure
print('=== LC25000 structure ===')
for p in sorted(LC25000_DIR.rglob('*')):
    if p.is_dir():
        count = len(list(p.glob('*.jpeg')) + list(p.glob('*.jpg')) + list(p.glob('*.png')))
        if count > 0:
            print(f'  {p.name}: {count} images')

print('\n=== Colorectal MNIST structure ===')
for p in sorted(COLORECTAL_DIR.rglob('*')):
    if p.is_dir():
        count = len(list(p.glob('*.tif')) + list(p.glob('*.png')) + list(p.glob('*.jpg')))
        if count > 0:
            print(f'  {p.name}: {count} images')

## Step 4 — Locate Dataset Paths


In [ ]:
# ---- Fix LC25000: point to parent of both colon and lung sets ----
lc25000_root = None

# Search for the parent folder containing both colon_image_sets and lung_image_sets
for p in LC25000_DIR.rglob('colon_aca'):
    if p.is_dir():
        # Go up two levels: colon_aca -> colon_image_sets -> lung_colon_image_set
        lc25000_root = p.parent.parent
        break

if lc25000_root:
    print(f'LC25000 root found: {lc25000_root}')
    for cls in LC25000_CLASSES:
        # Search recursively for each class folder
        cls_path = None
        for p in lc25000_root.rglob(cls):
            if p.is_dir():
                cls_path = p
                break
        count = len(list(cls_path.glob('*.jpeg')) +
                    list(cls_path.glob('*.jpg')) +
                    list(cls_path.glob('*.png'))) if cls_path else 0
        print(f'  {cls}: {count} images  (path: {cls_path})')
else:
    print('LC25000 root not found.')

# Store individual class paths for later use
LC25000_CLASS_PATHS = {}
for cls in LC25000_CLASSES:
    for p in LC25000_DIR.rglob(cls):
        if p.is_dir():
            LC25000_CLASS_PATHS[cls] = p
            break

print()

# ---- Fix Colorectal MNIST: handle numbered folder names ----
# Map numbered folder names to clean class names
COLORECTAL_FOLDER_MAP = {
    'tumor'  : ['01_TUMOR',  'TUMOR',  'tumor'],
    'mucosa' : ['06_MUCOSA', 'MUCOSA', 'mucosa'],
    'stroma' : ['02_STROMA', 'STROMA', 'stroma'],
    'complex': ['03_COMPLEX','COMPLEX','complex'],
    'lympho' : ['04_LYMPHO', 'LYMPHO', 'lympho'],
    'debris' : ['05_DEBRIS', 'DEBRIS', 'debris'],
    'adipose': ['07_ADIPOSE','ADIPOSE','adipose'],
    'empty'  : ['08_EMPTY',  'EMPTY',  'empty'],
}

colorectal_root = None
COLORECTAL_CLASS_PATHS = {}

# Find the base directory of colorectal dataset
for cls_name, possible_folders in COLORECTAL_FOLDER_MAP.items():
    for folder in possible_folders:
        for p in COLORECTAL_DIR.rglob(folder):
            if p.is_dir():
                COLORECTAL_CLASS_PATHS[cls_name] = p
                if colorectal_root is None:
                    colorectal_root = p.parent
                break
    if colorectal_root:
        break

# Find all class paths now that we know the root
for cls_name, possible_folders in COLORECTAL_FOLDER_MAP.items():
    if cls_name not in COLORECTAL_CLASS_PATHS:
        for folder in possible_folders:
            p = colorectal_root / folder
            if p.exists():
                COLORECTAL_CLASS_PATHS[cls_name] = p
                break

print(f'Colorectal MNIST root found: {colorectal_root}')
for cls_name, possible_folders in COLORECTAL_FOLDER_MAP.items():
    cls_path = COLORECTAL_CLASS_PATHS.get(cls_name)
    if cls_path:
        count = len(list(cls_path.glob('*.tif')) +
                    list(cls_path.glob('*.png')) +
                    list(cls_path.glob('*.jpg')))
        used = '← USED' if cls_name in COLORECTAL_CLASSES_MAPPED else '← excluded'
        print(f'  {cls_name} (folder {cls_path.name}): {count} images  {used}')
    else:
        print(f'  {cls_name}: NOT FOUND')

## Step 5 — Data Exploration & Visualisation

Check class distributions and sample images.

In [ ]:
# ---- Plot class distributions using the corrected class paths ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- LC25000 distribution (use LC25000_CLASS_PATHS from fixed Step 4) ---
lc_counts = {}
for cls in LC25000_CLASSES:
    cls_path = LC25000_CLASS_PATHS.get(cls)
    if cls_path and cls_path.exists():
        lc_counts[cls] = len(
            list(cls_path.glob('*.jpeg')) +
            list(cls_path.glob('*.jpg')) +
            list(cls_path.glob('*.png'))
        )
    else:
        lc_counts[cls] = 0

colors_lc = ['#4C72B0' if 'colon' in c else '#C44E52' for c in lc_counts.keys()]
axes[0].bar(lc_counts.keys(), lc_counts.values(), color=colors_lc,
            edgecolor='white', linewidth=0.5)
axes[0].set_title('LC25000 — Class Distribution (Training)', fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Image Count')
axes[0].tick_params(axis='x', rotation=15)
colon_patch = mpatches.Patch(color='#4C72B0', label='Colon (used in eval)')
lung_patch  = mpatches.Patch(color='#C44E52', label='Lung (train only)')
axes[0].legend(handles=[colon_patch, lung_patch])

# Verify counts printed
print('LC25000 counts:')
for cls, count in lc_counts.items():
    print(f'  {cls}: {count}')

# --- Colorectal MNIST distribution (use COLORECTAL_CLASS_PATHS from fixed Step 4) ---
cr_counts = {}
for cls_name in COLORECTAL_FOLDER_MAP.keys():
    cls_path = COLORECTAL_CLASS_PATHS.get(cls_name)
    if cls_path and cls_path.exists():
        cr_counts[cls_name] = len(
            list(cls_path.glob('*.tif')) +
            list(cls_path.glob('*.png')) +
            list(cls_path.glob('*.jpg'))
        )
    else:
        cr_counts[cls_name] = 0

colors_cr = ['#2ecc71' if cls in COLORECTAL_CLASSES_MAPPED else '#bdc3c7'
             for cls in cr_counts.keys()]
axes[1].bar(cr_counts.keys(), cr_counts.values(), color=colors_cr,
            edgecolor='white', linewidth=0.5)
axes[1].set_title('Colorectal MNIST — Class Distribution (Test)', fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Image Count')
axes[1].tick_params(axis='x', rotation=30)
used_patch = mpatches.Patch(color='#2ecc71', label='Used in evaluation')
excl_patch = mpatches.Patch(color='#bdc3c7', label='Excluded (no LC25000 equivalent)')
axes[1].legend(handles=[used_patch, excl_patch])

print('\nColorectal MNIST counts:')
for cls, count in cr_counts.items():
    print(f'  {cls}: {count}')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'class_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nClass distribution plot saved.')

In [ ]:
def show_samples_from_paths(class_paths, class_names, title, n_per_class=4):
    """
    Show sample images using pre-resolved path dictionary.
    class_paths: dict of {class_name: Path}
    class_names: list of class names to show
    """
    ext_list = ['*.jpg', '*.jpeg', '*.png', '*.tif']

    fig, axes = plt.subplots(len(class_names), n_per_class,
                              figsize=(n_per_class * 3, len(class_names) * 3))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)

    for i, cls in enumerate(class_names):
        cls_path = class_paths.get(cls)

        # Collect images
        imgs = []
        if cls_path and cls_path.exists():
            for ext in ext_list:
                imgs.extend(list(cls_path.glob(ext)))

        print(f'  {cls}: found {len(imgs)} images at {cls_path}')
        imgs = imgs[:n_per_class]

        for j in range(n_per_class):
            ax = axes[i][j] if len(class_names) > 1 else axes[j]
            if j < len(imgs):
                try:
                    img = Image.open(imgs[j]).convert('RGB')
                    ax.imshow(img)
                    ax.axis('off')
                    if j == 0:
                        ax.set_title(cls, fontsize=10, fontweight='bold', loc='left')
                except Exception as e:
                    ax.axis('off')
                    ax.text(0.5, 0.5, f'Error:\n{e}', ha='center', va='center',
                            fontsize=7, transform=ax.transAxes)
            else:
                ax.axis('off')
                ax.text(0.5, 0.5, 'No image', ha='center', va='center',
                        fontsize=9, color='gray', transform=ax.transAxes)

    plt.tight_layout()
    save_name = title[:20].replace(' ', '_').replace('—', '').strip('_')
    plt.savefig(RESULTS_DIR / f'samples_{save_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


# Show LC25000 colon samples
print('LC25000 colon samples:')
show_samples_from_paths(
    LC25000_CLASS_PATHS,
    ['colon_aca', 'colon_n'],
    'LC25000 — Colon Classes (Training)'
)

# Show Colorectal MNIST mapped class samples
print('\nColorectal MNIST mapped samples:')
show_samples_from_paths(
    COLORECTAL_CLASS_PATHS,
    ['tumor', 'mucosa'],
    'Colorectal MNIST — Mapped Classes (Test)'
)

In [ ]:
# ---- Pixel size check for both datasets ----

print('=' * 55)
print('LC25000 — Sample Image Size Check')
print('=' * 55)
for cls in LC25000_CLASSES:
    cls_path = LC25000_CLASS_PATHS.get(cls)
    if cls_path and cls_path.exists():
        samples = (list(cls_path.glob('*.jpeg')) +
                   list(cls_path.glob('*.jpg')) +
                   list(cls_path.glob('*.png')))
        if samples:
            img = Image.open(samples[0]).convert('RGB')
            print(f'  {cls:12s} → size: {img.size}  mode: {img.mode}  format: {Path(samples[0]).suffix}')
        else:
            print(f'  {cls:12s} → no images found')
    else:
        print(f'  {cls:12s} → path not found')

print()
print('=' * 55)
print('Colorectal MNIST — Sample Image Size Check')
print('=' * 55)
for cls_name in COLORECTAL_FOLDER_MAP.keys():
    cls_path = COLORECTAL_CLASS_PATHS.get(cls_name)
    if cls_path and cls_path.exists():
        samples = (list(cls_path.glob('*.tif')) +
                   list(cls_path.glob('*.png')) +
                   list(cls_path.glob('*.jpg')))
        if samples:
            img = Image.open(samples[0]).convert('RGB')
            used = '← USED' if cls_name in COLORECTAL_CLASSES_MAPPED else ''
            print(f'  {cls_name:10s} → size: {img.size}  mode: {img.mode}  format: {Path(samples[0]).suffix}  {used}')
        else:
            print(f'  {cls_name:10s} → no images found')
    else:
        print(f'  {cls_name:10s} → path not found')

print()
print('=' * 55)
print('INTERPRETATION')
print('=' * 55)
# Check colorectal sizes specifically
colorectal_sizes = []
for cls_name in COLORECTAL_CLASSES_MAPPED:
    cls_path = COLORECTAL_CLASS_PATHS.get(cls_name)
    if cls_path:
        samples = (list(cls_path.glob('*.tif')) +
                   list(cls_path.glob('*.png')) +
                   list(cls_path.glob('*.jpg')))
        if samples:
            img = Image.open(samples[0])
            colorectal_sizes.append(img.size)

if colorectal_sizes:
    w, h = colorectal_sizes[0]
    if w <= 150 and h <= 150:
        print(f'  Colorectal images are {w}x{h}px — SMALL version (150x150).')
        print('  No extra processing needed. Resize to 224x224 is fine.')
    elif w >= 5000 or h >= 5000:
        print(f'  Colorectal images are {w}x{h}px — LARGE tile version (5000x5000).')
        print('  WARNING: You need center-crop or tiling before resizing.')
        print('  The notebook transform pipeline needs to be updated.')
    else:
        print(f'  Colorectal images are {w}x{h}px — intermediate size.')
        print('  Resize to 224x224 should still work fine.')



## Step 6 — Define Data Transforms

We define two separate transform pipelines:
- **Training transforms**: augmentation (flip, rotation, color jitter) to improve generalisation
- **Validation/Test transforms**: only resize and normalize — no augmentation on evaluation data

Both use ImageNet mean/std normalization since we use a pretrained backbone (ResNet50).

In [ ]:
IMG_SIZE = 224  # Standard for ResNet/EfficientNet

# ImageNet normalization stats (used because we load pretrained weights)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print('Transforms defined:')
print('  Training  :', [type(t).__name__ for t in train_transforms.transforms])
print('  Val/Test  :', [type(t).__name__ for t in val_test_transforms.transforms])

## Step 7 — Build LC25000 DataLoaders (Train / Validation Split)

We use an 80/20 train-validation split on LC25000.  
All 5 classes are included for training.

In [ ]:
from torch.utils.data import random_split

# Load full LC25000 dataset (all 5 classes)
full_lc_dataset = ImageFolder(root=str(lc25000_root), transform=train_transforms)

print('LC25000 class mapping (index → label):')
for label, idx in full_lc_dataset.class_to_idx.items():
    print(f'  {idx}: {label}')

LC25000_CLASS_TO_IDX = full_lc_dataset.class_to_idx
LC25000_IDX_TO_CLASS = {v: k for k, v in LC25000_CLASS_TO_IDX.items()}
NUM_CLASSES = len(LC25000_CLASS_TO_IDX)
print(f'\nTotal images: {len(full_lc_dataset)}')
print(f'Number of classes: {NUM_CLASSES}')

# 80/20 split
train_size = int(0.8 * len(full_lc_dataset))
val_size   = len(full_lc_dataset) - train_size
train_dataset, val_dataset = random_split(
    full_lc_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply val transforms to val split (no augmentation)
# We need a wrapper since random_split doesn't let us change transforms
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # img is already a tensor from train_transforms — reload from path
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

val_dataset_clean = TransformSubset(val_dataset, val_test_transforms)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset_clean, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'\nTrain batches: {len(train_loader)} ({train_size} images)')
print(f'Val batches  : {len(val_loader)} ({val_size} images)')

In [ ]:
from torch.utils.data import random_split, ConcatDataset
import tempfile
import os

# ---- Create a flat unified folder structure for ImageFolder ----
# We symlink all 5 class folders into one unified directory
# so ImageFolder can read all classes from a single root

UNIFIED_DIR = BASE_DIR / 'lc25000_unified'
UNIFIED_DIR.mkdir(exist_ok=True)

print('Building unified LC25000 folder structure...')
for cls, cls_path in LC25000_CLASS_PATHS.items():
    link_path = UNIFIED_DIR / cls
    # Remove existing symlink if it exists
    if link_path.exists() or link_path.is_symlink():
        link_path.unlink()
    os.symlink(cls_path, link_path)
    count = len(list(cls_path.glob('*.jpeg')) +
                list(cls_path.glob('*.jpg')) +
                list(cls_path.glob('*.png')))
    print(f'  Linked: {cls} → {cls_path} ({count} images)')

# Verify unified structure
print(f'\nUnified root: {UNIFIED_DIR}')
for p in sorted(UNIFIED_DIR.iterdir()):
    print(f'  {p.name} (symlink={p.is_symlink()})')

# ---- Load with ImageFolder from unified root ----
full_lc_dataset = ImageFolder(root=str(UNIFIED_DIR), transform=train_transforms)

print('\nLC25000 class mapping (index → label):')
for label, idx in full_lc_dataset.class_to_idx.items():
    print(f'  {idx}: {label}')

LC25000_CLASS_TO_IDX = full_lc_dataset.class_to_idx
LC25000_IDX_TO_CLASS = {v: k for k, v in LC25000_CLASS_TO_IDX.items()}
NUM_CLASSES = len(LC25000_CLASS_TO_IDX)
print(f'\nTotal images : {len(full_lc_dataset)}')
print(f'Num classes  : {NUM_CLASSES}')

# ---- 80/20 train/val split ----
train_size = int(0.8 * len(full_lc_dataset))
val_size   = len(full_lc_dataset) - train_size
train_dataset, val_dataset = random_split(
    full_lc_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply clean val transforms (no augmentation)
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

val_dataset_clean = TransformSubset(val_dataset, val_test_transforms)

BATCH_SIZE   = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset_clean, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'\nTrain batches : {len(train_loader)} ({train_size} images)')
print(f'Val batches   : {len(val_loader)} ({val_size} images)')

# Sanity check — verify one batch loads correctly
images, labels = next(iter(train_loader))
print(f'\nSanity check batch:')
print(f'  Image tensor shape : {images.shape}')
print(f'  Labels             : {labels[:8].tolist()}')
print(f'  Label names        : {[LC25000_IDX_TO_CLASS[l.item()] for l in labels[:8]]}')

## Step 8 — Build the Model (ResNet50 with Pretrained Weights)

We use **ResNet50** pretrained on ImageNet as our backbone.  
Only the final classification head is replaced to match our 5-class task.  
The feature extraction layers are fine-tuned (not frozen during training).

In [ ]:
def build_model(num_classes, freeze_backbone=False):
    """
    Build ResNet50 with custom classification head.
    freeze_backbone: if True, only trains the final FC layer.
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace final fully connected layer
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes)
    )

    return model

# Build model — full fine-tuning (freeze_backbone=False)
model = build_model(num_classes=NUM_CLASSES, freeze_backbone=False)
model = model.to(device)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model head          : {model.fc}')

## Step 9 — Training Loop

We train with:
- **CrossEntropyLoss** (standard for multi-class classification)
- **Adam optimizer** with weight decay (L2 regularisation)
- **ReduceLROnPlateau** scheduler (reduces LR when val loss plateaus)
- **Early stopping** to prevent overfitting
- **Best model checkpointing** based on validation accuracy

In [ ]:
# ---- Training configuration ----
NUM_EPOCHS    = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 5   # Early stopping patience

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
    return running_loss / total, correct / total

print('Training configuration ready.')
print(f'  Epochs       : {NUM_EPOCHS}')
print(f'  Learning rate: {LEARNING_RATE}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Early stop   : patience={PATIENCE}')

In [ ]:
# ---- Run training ----
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc  = 0.0
patience_ctr  = 0
BEST_MODEL_PATH = MODEL_DIR / 'best_lc25000_model.pth'

print(f'Training on {device}...\n')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7}')
print('-' * 55)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = validate_epoch(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.4f} | {val_loss:>8.4f} | {val_acc:>7.4f}')

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'class_to_idx': LC25000_CLASS_TO_IDX
        }, BEST_MODEL_PATH)
        patience_ctr = 0
        print(f'         ↑ New best model saved (val_acc={val_acc:.4f})')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

print(f'\nTraining complete. Best validation accuracy: {best_val_acc:.4f}')

## Step 10 — Plot Training History

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_ran, history['train_loss'], label='Train loss', color='#4C72B0', linewidth=2)
axes[0].plot(epochs_ran, history['val_loss'],   label='Val loss',   color='#C44E52', linewidth=2, linestyle='--')
axes[0].set_title('Training vs Validation Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, [a*100 for a in history['train_acc']], label='Train acc', color='#4C72B0', linewidth=2)
axes[1].plot(epochs_ran, [a*100 for a in history['val_acc']],   label='Val acc',   color='#C44E52', linewidth=2, linestyle='--')
axes[1].set_title('Training vs Validation Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training history plot saved.')

In [ ]:
print(type(COLORECTAL_CLASS_PATHS))
print(COLORECTAL_CLASS_PATHS)

In [ ]:
# Force delete the wrongly named class and rebuild as proper dictionary
import importlib

# Delete the corrupted variable
if 'COLORECTAL_CLASS_PATHS' in dir():
    del COLORECTAL_CLASS_PATHS

from pathlib import Path

# Rebuild as a plain dictionary
COLORECTAL_CLASS_PATHS = {}

COLORECTAL_DIR = Path('/content/cancer_detection/colorectal')

COLORECTAL_FOLDER_MAP = {
    'tumor'  : ['01_TUMOR',  'TUMOR',  'tumor'],
    'mucosa' : ['06_MUCOSA', 'MUCOSA', 'mucosa'],
    'stroma' : ['02_STROMA', 'STROMA', 'stroma'],
    'complex': ['03_COMPLEX','COMPLEX','complex'],
    'lympho' : ['04_LYMPHO', 'LYMPHO', 'lympho'],
    'debris' : ['05_DEBRIS', 'DEBRIS', 'debris'],
    'adipose': ['07_ADIPOSE','ADIPOSE','adipose'],
    'empty'  : ['08_EMPTY',  'EMPTY',  'empty'],
}

for cls_name, possible_folders in COLORECTAL_FOLDER_MAP.items():
    for folder in possible_folders:
        for p in COLORECTAL_DIR.rglob(folder):
            if p.is_dir():
                COLORECTAL_CLASS_PATHS[cls_name] = p
                break
        if cls_name in COLORECTAL_CLASS_PATHS:
            break

# Verify it is now a dict
print(f'Type: {type(COLORECTAL_CLASS_PATHS)}')
print(f'Is dict: {isinstance(COLORECTAL_CLASS_PATHS, dict)}')
print()
print('Paths:')
for cls, path in COLORECTAL_CLASS_PATHS.items():
    count = len(list(path.glob('*.tif')) +
                list(path.glob('*.png')) +
                list(path.glob('*.jpg'))) if path.exists() else 0
    print(f'  {cls}: {path.name} — {count} images')

## Step 11 — Class Mapping & Colorectal MNIST Test Dataset

This is the core of the cross-dataset methodology.  

We build a **custom dataset** that:
1. Only loads `tumor` and `mucosa` images from Colorectal MNIST
2. Maps their labels to the equivalent LC25000 class indices
3. Applies the same val/test transforms (resize + normalize)

**Class mapping applied:**
- `tumor` → `colon_aca` index (malignant)
- `mucosa` → `colon_n` index (benign/normal)

In [ ]:
class ColorectalMappedDataset(Dataset):
    """
    Loads only tumor and mucosa images from Colorectal Histology MNIST.
    Maps labels to LC25000 class indices for cross-dataset evaluation.

    Mapping:
        tumor  → colon_aca (index 0) — malignant
        mucosa → colon_n   (index 1) — benign
    """
    def __init__(self, class_paths, lc25000_class_to_idx, transform=None):
        self.transform = transform
        self.samples   = []

        LABEL_MAP = {
            'tumor' : 'colon_aca',
            'mucosa': 'colon_n'
        }

        print('Class mapping applied:')
        for cr_cls, lc_cls in LABEL_MAP.items():
            lc_idx   = lc25000_class_to_idx[lc_cls]

            # Use the dictionary directly — no path joining needed
            cls_path = class_paths.get(cr_cls)

            if cls_path is None or not cls_path.exists():
                print(f'  WARNING: path not found for "{cr_cls}" — skipping')
                continue

            img_files = (list(cls_path.glob('*.tif')) +
                         list(cls_path.glob('*.png')) +
                         list(cls_path.glob('*.jpg')) +
                         list(cls_path.glob('*.jpeg')))

            for f in img_files:
                self.samples.append((str(f), lc_idx, cr_cls))

            print(f'  Colorectal "{cr_cls}" ({cls_path.name}) '
                  f'→ LC25000 "{lc_cls}" (idx={lc_idx}) | {len(img_files)} images')

        print(f'\nTotal test images after mapping: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, cr_class = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

    def get_original_labels(self):
        return [s[2] for s in self.samples]


# Build mapped test dataset
colorectal_test_dataset = ColorectalMappedDataset(
    class_paths          = COLORECTAL_CLASS_PATHS,
    lc25000_class_to_idx = LC25000_CLASS_TO_IDX,
    transform            = val_test_transforms
)

colorectal_test_loader = DataLoader(
    colorectal_test_dataset,
    batch_size  = 32,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = True
)

# Sanity check
print(f'\nTest loader batches: {len(colorectal_test_loader)}')
images, labels = next(iter(colorectal_test_loader))
print(f'Batch shape : {images.shape}')
print(f'Labels seen : {labels[:8].tolist()}')
print(f'Label names : {[LC25000_IDX_TO_CLASS[l.item()] for l in labels[:8]]}')

## Step 12 — Load Best Model & Run Cross-Dataset Inference

We load the best LC25000-trained model and run it on Colorectal MNIST.  
**No retraining. No fine-tuning. Weights are fully frozen.**  
This is a pure generalization test.

In [ ]:
# Load best saved model
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Best model loaded from epoch {checkpoint['epoch']} "
      f"(LC25000 val_acc = {checkpoint['val_acc']:.4f})")

# Freeze all weights — inference only
for param in model.parameters():
    param.requires_grad = False
model.eval()
print('Model weights frozen. Running cross-dataset inference...')

# ---- Collect predictions and probabilities ----
all_labels = []
all_preds  = []
all_probs  = []

with torch.no_grad():
    for images, labels in colorectal_test_loader:
        images = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

print(f'Inference complete. {len(all_labels)} images evaluated.')

## Step 13 — Evaluation Metrics

We evaluate using:
- **Accuracy** — overall correct predictions
- **F1 Score** — harmonic mean of precision and recall (handles class imbalance better)
- **AUC-ROC** — discrimination ability (only meaningful for the 2 mapped classes)
- **Confusion Matrix** — shows where the model confuses classes
- **Per-class report** — precision, recall, F1 per class

In [ ]:
# We only evaluate on the 2 mapped classes
# Get the indices for colon_aca and colon_n from the LC25000 mapping
colon_aca_idx = LC25000_CLASS_TO_IDX['colon_aca']
colon_n_idx   = LC25000_CLASS_TO_IDX['colon_n']
MAPPED_INDICES = [colon_aca_idx, colon_n_idx]
MAPPED_NAMES   = ['colon_aca (cancer)', 'colon_n (benign)']

# Filter to only predictions that the model assigned to one of the mapped classes
# (All test samples SHOULD be mapped, but we verify)
mask = np.isin(all_labels, MAPPED_INDICES)
y_true = all_labels[mask]
y_pred = all_preds[mask]
y_prob = all_probs[mask]

# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
f1       = f1_score(y_true, y_pred, average='weighted')

# AUC-ROC (binary: colon_aca vs colon_n)
# Use probability of colon_aca as positive class
y_prob_cancer = y_prob[:, colon_aca_idx]
y_binary      = (y_true == colon_aca_idx).astype(int)
auc_roc       = roc_auc_score(y_binary, y_prob_cancer)

print('=' * 55)
print('  CROSS-DATASET EVALUATION RESULTS')
print('  LC25000 → Colorectal Histology MNIST')
print('=' * 55)
print(f'  Samples evaluated : {len(y_true)}')
print(f'  Accuracy          : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  Weighted F1 Score : {f1:.4f}')
print(f'  AUC-ROC           : {auc_roc:.4f}')
print('=' * 55)

print('\nPer-class report:')
target_names_report = {
    colon_aca_idx: 'colon_aca→tumor',
    colon_n_idx:   'colon_n→mucosa'
}
report_labels = sorted(MAPPED_INDICES)
report_names  = [target_names_report[i] for i in report_labels]
print(classification_report(y_true, y_pred, labels=report_labels, target_names=report_names))

In [ ]:
# Check what the model is actually predicting
import collections
pred_counts = collections.Counter(all_preds.tolist())
print('Prediction distribution:')
for idx, count in sorted(pred_counts.items()):
    print(f'  {LC25000_IDX_TO_CLASS[idx]}: {count} times')

print(f'\nTrue label distribution:')
true_counts = collections.Counter(all_labels.tolist())
for idx, count in sorted(true_counts.items()):
    print(f'  {LC25000_IDX_TO_CLASS[idx]}: {count} times')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- What the model actually predicted ---
pred_classes = [LC25000_IDX_TO_CLASS[i] for i in sorted(pred_counts.keys())]
pred_values  = [pred_counts[i] for i in sorted(pred_counts.keys())]
colors = ['#4C72B0' if 'colon' in c else '#C44E52' for c in pred_classes]

bars = axes[0].bar(pred_classes, pred_values, color=colors,
                   edgecolor='white', linewidth=0.5)
axes[0].set_title('What the Model Predicted\n(on Colorectal MNIST test images)',
                  fontweight='bold')
axes[0].set_xlabel('Predicted Class')
axes[0].set_ylabel('Number of Predictions')
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, pred_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', fontsize=10, fontweight='bold')

colon_patch = mpatches.Patch(color='#4C72B0', label='Colon class (correct domain)')
lung_patch  = mpatches.Patch(color='#C44E52', label='Lung class (wrong domain!)')
axes[0].legend(handles=[colon_patch, lung_patch])
axes[0].axhline(y=625, color='green', linestyle='--', linewidth=1.5,
                label='Expected (625 per class)')

# --- True vs predicted for colon classes only ---
categories = ['colon_aca\n(cancer)', 'colon_n\n(benign)']
true_vals  = [625, 625]
pred_vals_colon = [
    pred_counts.get(LC25000_CLASS_TO_IDX['colon_aca'], 0),
    pred_counts.get(LC25000_CLASS_TO_IDX['colon_n'], 0)
]
lung_wrongly = [
    1250 - pred_counts.get(LC25000_CLASS_TO_IDX['colon_aca'], 0)
         - pred_counts.get(LC25000_CLASS_TO_IDX['colon_n'], 0),
    0
]

x = np.arange(len(categories))
width = 0.3
axes[1].bar(x - width, true_vals, width, label='True count',
            color='#2ecc71', edgecolor='white')
axes[1].bar(x, pred_vals_colon, width, label='Predicted as colon',
            color='#4C72B0', edgecolor='white')
axes[1].set_title('True Count vs Colon Predictions\n(domain shift impact)',
                  fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Annotate lung_scc dominance
axes[1].annotate(f'833 images wrongly\npredicted as lung_scc',
                 xy=(0.5, 100), xytext=(0.8, 400),
                 fontsize=9, color='#C44E52',
                 arrowprops=dict(arrowstyle='->', color='#C44E52'))

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'prediction_distribution_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Print the key finding clearly
total = len(all_preds)
lung_preds = sum(pred_counts.get(LC25000_CLASS_TO_IDX[c], 0)
                 for c in ['lung_aca', 'lung_n', 'lung_scc'])
print('=' * 55)
print('KEY FINDING — PREDICTION DISTRIBUTION')
print('=' * 55)
print(f'Total test images        : {total}')
print(f'Predicted as LUNG class  : {lung_preds} ({lung_preds/total*100:.1f}%)')
print(f'Predicted as COLON class : {total-lung_preds} ({(total-lung_preds)/total*100:.1f}%)')
print(f'All test images are colon tissue — lung predictions are all WRONG')
print(f'Dominant wrong class     : lung_scc ({pred_counts[LC25000_CLASS_TO_IDX["lung_scc"]]} predictions)')
print('=' * 55)

## Step 14 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=MAPPED_INDICES)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted: cancer\n(colon_aca)', 'Predicted: benign\n(colon_n)'],
    yticklabels=['Actual: tumor\n(cancer)', 'Actual: mucosa\n(benign)'],
    ax=ax, linewidths=0.5, linecolor='gray'
)
ax.set_title('Confusion Matrix — Cross-Dataset Evaluation\nLC25000 model on Colorectal MNIST', fontweight='bold', pad=15)
ax.set_ylabel('True Label (Colorectal MNIST)', fontweight='bold')
ax.set_xlabel('Predicted Label (LC25000 classes)', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix_cross_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpret the confusion matrix
TP = cm[0,0]  # Predicted cancer, actually cancer (tumor)
FN = cm[0,1]  # Predicted benign, actually cancer — dangerous!
FP = cm[1,0]  # Predicted cancer, actually benign
TN = cm[1,1]  # Predicted benign, actually benign (mucosa)
print(f'True Positives (cancer correctly identified): {TP}')
print(f'False Negatives (cancer missed — critical!): {FN}')
print(f'False Positives (benign wrongly flagged)   : {FP}')
print(f'True Negatives  (benign correctly passed)  : {TN}')
sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
print(f'\nSensitivity (cancer recall): {sensitivity:.4f}')
print(f'Specificity (benign recall) : {specificity:.4f}')

## Step 15 — ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_binary, y_prob_cancer)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#4C72B0', linewidth=2.5,
        label=f'ROC Curve (AUC = {auc_roc:.4f})')
ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Random classifier (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#4C72B0')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('ROC Curve — Cross-Dataset Evaluation\n(Cancer vs Benign on Colorectal MNIST)', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_curve_cross_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 16 — Compare LC25000 vs Cross-Dataset Performance

This comparison is the core finding of your research.  
It directly shows the effect of domain shift.

In [ ]:
# LC25000 validation accuracy (from training)
lc25000_val_acc = checkpoint['val_acc']

# Build comparison table
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score (weighted)', 'AUC-ROC'],
    'LC25000 (Same-dataset val)': [
        f'{lc25000_val_acc:.4f}',
        'N/A (see training logs)',
        'N/A (see training logs)'
    ],
    'Colorectal MNIST (Cross-dataset)': [
        f'{accuracy:.4f}',
        f'{f1:.4f}',
        f'{auc_roc:.4f}'
    ]
})
print('=== Performance Comparison ===')
print(comparison.to_string(index=False))

# Visual comparison bar chart
fig, ax = plt.subplots(figsize=(8, 5))
bars_x     = [0, 1]
bars_h     = [lc25000_val_acc * 100, accuracy * 100]
bar_colors = ['#4C72B0', '#2ecc71']
bars = ax.bar(bars_x, bars_h, color=bar_colors, width=0.4,
              edgecolor='white', linewidth=0.5)
ax.set_xticks(bars_x)
ax.set_xticklabels([
    'LC25000\n(same-dataset validation)',
    'Colorectal MNIST\n(cross-dataset test)'
], fontsize=11)
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title('Same-Dataset vs Cross-Dataset Performance\n(Domain Shift Effect)', fontweight='bold')
ax.set_ylim([0, 110])
for bar, val in zip(bars, bars_h):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.2f}%', ha='center', fontweight='bold', fontsize=12)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
drop = lc25000_val_acc * 100 - accuracy * 100
ax.annotate(f'Domain shift drop:\n{drop:.2f}%',
            xy=(1, accuracy*100), xytext=(1.3, (lc25000_val_acc*100 + accuracy*100)/2),
            fontsize=10, color='#C44E52',
            arrowprops=dict(arrowstyle='->', color='#C44E52'))
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Restricts Prediction

In [ ]:
# Restrict model output to only colon classes during evaluation
COLON_INDICES = [LC25000_CLASS_TO_IDX['colon_aca'],
                 LC25000_CLASS_TO_IDX['colon_n']]

all_labels_filtered = []
all_preds_filtered  = []
all_probs_filtered  = []

model.eval()
with torch.no_grad():
    for images, labels in colorectal_test_loader:
        images = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)

        # Zero out lung class probabilities
        # Force model to only choose between colon_aca and colon_n
        colon_probs = probs[:, COLON_INDICES]
        colon_probs = colon_probs / colon_probs.sum(dim=1, keepdim=True)

        _, colon_pred_idx = colon_probs.max(1)
        final_preds = torch.tensor([COLON_INDICES[i] for i in colon_pred_idx])

        all_labels_filtered.extend(labels.numpy())
        all_preds_filtered.extend(final_preds.numpy())
        all_probs_filtered.extend(colon_probs.cpu().numpy())

all_labels_filtered = np.array(all_labels_filtered)
all_preds_filtered  = np.array(all_preds_filtered)
all_probs_filtered  = np.array(all_probs_filtered)

# Rerun metrics on filtered predictions
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

accuracy_f = accuracy_score(all_labels_filtered, all_preds_filtered)
f1_f       = f1_score(all_labels_filtered, all_preds_filtered, average='weighted')

y_binary_f      = (all_labels_filtered == LC25000_CLASS_TO_IDX['colon_aca']).astype(int)
y_prob_cancer_f = all_probs_filtered[:, 0]  # index 0 = colon_aca in COLON_INDICES
auc_f           = roc_auc_score(y_binary_f, y_prob_cancer_f)

print('=' * 55)
print('RESTRICTED EVALUATION (colon classes only)')
print('=' * 55)
print(f'Accuracy   : {accuracy_f:.4f} ({accuracy_f*100:.2f}%)')
print(f'F1 Score   : {f1_f:.4f}')
print(f'AUC-ROC    : {auc_f:.4f}')
print('=' * 55)

pred_counts_f = {LC25000_IDX_TO_CLASS[i]:
                 list(all_preds_filtered).count(i) for i in COLON_INDICES}
print('Prediction distribution (restricted):')
for cls, count in pred_counts_f.items():
    print(f'  {cls}: {count}')

In [ ]:
# ============================================================
# RESTRICTED EVALUATION — Full Visualisation
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report, roc_curve
import matplotlib.patches as mpatches

COLON_LABEL_NAMES = ['colon_aca (cancer)', 'colon_n (benign)']
COLON_INDICES     = [LC25000_CLASS_TO_IDX['colon_aca'],
                     LC25000_CLASS_TO_IDX['colon_n']]

# ---- 1. Prediction Distribution Comparison ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Restricted Evaluation — Colon Classes Only',
             fontweight='bold', fontsize=14)

# Unrestricted predictions
all_classes  = [LC25000_IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]
unres_counts = [list(all_preds).count(LC25000_CLASS_TO_IDX[c])
                for c in all_classes]
colors_unres = ['#4C72B0' if 'colon' in c else '#C44E52' for c in all_classes]
bars = axes[0].bar(all_classes, unres_counts, color=colors_unres,
                   edgecolor='white', linewidth=0.5)
axes[0].set_title('Unrestricted Predictions\n(all 5 classes allowed)',
                  fontweight='bold')
axes[0].set_xlabel('Predicted Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, unres_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 5, str(val),
                 ha='center', fontsize=9, fontweight='bold')
colon_p = mpatches.Patch(color='#4C72B0', label='Colon class')
lung_p  = mpatches.Patch(color='#C44E52', label='Lung class (wrong domain)')
axes[0].legend(handles=[colon_p, lung_p])

# Restricted predictions
res_labels = ['colon_aca\n(cancer)', 'colon_n\n(benign)']
res_counts = [list(all_preds_filtered).count(COLON_INDICES[0]),
              list(all_preds_filtered).count(COLON_INDICES[1])]
bars2 = axes[1].bar(res_labels, res_counts, color=['#4C72B0', '#2ecc71'],
                    edgecolor='white', linewidth=0.5, width=0.4)
axes[1].set_title('Restricted Predictions\n(colon classes only)',
                  fontweight='bold')
axes[1].set_xlabel('Predicted Class')
axes[1].set_ylabel('Count')
axes[1].axhline(y=625, color='gray', linestyle='--', linewidth=1.2,
                label='Expected (625 per class)')
axes[1].legend()
for bar, val in zip(bars2, res_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 5, str(val),
                 ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'restricted_prediction_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ---- 2. Confusion Matrix ----
cm_r = confusion_matrix(all_labels_filtered, all_preds_filtered,
                        labels=COLON_INDICES)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_r, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted: cancer\n(colon_aca)',
                 'Predicted: benign\n(colon_n)'],
    yticklabels=['Actual: tumor\n(cancer)',
                 'Actual: mucosa\n(benign)'],
    ax=ax, linewidths=0.5, linecolor='gray'
)
ax.set_title('Confusion Matrix — Restricted Evaluation\n'
             '(Colon classes only, no lung predictions allowed)',
             fontweight='bold', pad=15)
ax.set_ylabel('True Label (Colorectal MNIST)', fontweight='bold')
ax.set_xlabel('Predicted Label (LC25000 colon classes)', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'restricted_confusion_matrix.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Compute sensitivity and specificity from restricted CM
TP_r = cm_r[0, 0]
FN_r = cm_r[0, 1]
FP_r = cm_r[1, 0]
TN_r = cm_r[1, 1]
sensitivity_r = TP_r / (TP_r + FN_r) if (TP_r + FN_r) > 0 else 0
specificity_r = TN_r / (TN_r + FP_r) if (TN_r + FP_r) > 0 else 0

# ---- 3. ROC Curve ----
fpr_r, tpr_r, _ = roc_curve(y_binary_f, y_prob_cancer_f)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_r, tpr_r, color='#4C72B0', linewidth=2.5,
        label=f'Restricted ROC (AUC = {auc_f:.4f})')
ax.plot([0,1],[0,1], 'k--', linewidth=1,
        label='Random classifier (AUC = 0.5)')
ax.fill_between(fpr_r, tpr_r, alpha=0.1, color='#4C72B0')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('ROC Curve — Restricted Evaluation\n'
             '(Cancer vs Benign, colon classes only)',
             fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'restricted_roc_curve.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ---- 4. Side-by-side Metrics Comparison ----
fig, ax = plt.subplots(figsize=(10, 6))

metrics      = ['Accuracy', 'F1 Score', 'AUC-ROC']
unres_scores = [accuracy * 100, f1 * 100, auc_roc * 100]
res_scores   = [accuracy_f * 100, f1_f * 100, auc_f * 100]
lc_scores    = [lc25000_val_acc * 100, None, None]

x     = np.arange(len(metrics))
width = 0.25

bars_lc  = ax.bar(x - width, [lc25000_val_acc*100, 0, 0], width,
                  label='LC25000 (same-dataset)', color='#4C72B0',
                  edgecolor='white')
bars_un  = ax.bar(x, unres_scores, width,
                  label='Colorectal MNIST — unrestricted',
                  color='#C44E52', edgecolor='white')
bars_res = ax.bar(x + width, res_scores, width,
                  label='Colorectal MNIST — restricted (colon only)',
                  color='#2ecc71', edgecolor='white')

ax.set_ylabel('Score (%)', fontweight='bold')
ax.set_title('Performance Comparison\nLC25000 vs Cross-Dataset (Unrestricted vs Restricted)',
             fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim([0, 115])
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar in [*bars_lc, *bars_un, *bars_res]:
    h = bar.get_height()
    if h > 1:
        ax.text(bar.get_x() + bar.get_width()/2, h + 1.5,
                f'{h:.1f}%', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'full_metrics_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ---- 5. Print Full Summary ----
print('=' * 60)
print('  RESTRICTED EVALUATION SUMMARY')
print('=' * 60)
print(f'  Accuracy    : {accuracy_f:.4f} ({accuracy_f*100:.2f}%)')
print(f'  F1 Score    : {f1_f:.4f}')
print(f'  AUC-ROC     : {auc_f:.4f}')
print(f'  Sensitivity : {sensitivity_r:.4f}')
print(f'  Specificity : {specificity_r:.4f}')
print('=' * 60)
print(f'  TP (cancer correctly found) : {TP_r}')
print(f'  FN (cancer missed)          : {FN_r}')
print(f'  FP (benign wrongly flagged) : {FP_r}')
print(f'  TN (benign correctly passed): {TN_r}')
print('=' * 60)
print('\nClassification Report (Restricted):')
print(classification_report(
    all_labels_filtered,
    all_preds_filtered,
    labels=COLON_INDICES,
    target_names=['colon_aca→tumor', 'colon_n→mucosa']
))

# Save restricted summary
with open(RESULTS_DIR / 'restricted_results_summary.txt', 'w') as f:
    f.write(f"""
RESTRICTED CROSS-DATASET EVALUATION SUMMARY
Train: LC25000 (all 5 classes) | Test: Colorectal MNIST (colon only)
Lung class predictions zeroed out — model forced to choose between
colon_aca and colon_n only.
============================================================
Accuracy    : {accuracy_f:.4f} ({accuracy_f*100:.2f}%)
F1 Score    : {f1_f:.4f}
AUC-ROC     : {auc_f:.4f}
Sensitivity : {sensitivity_r:.4f}
Specificity : {specificity_r:.4f}
------------------------------------------------------------
TP : {TP_r}  |  FN : {FN_r}
FP : {FP_r}  |  TN : {TN_r}
============================================================
""")
print(f'\nAll restricted results saved to: {RESULTS_DIR}')

## Step 17 — Save Full Results Summary

In [ ]:
# ============================================================
# UPDATED FULL RESULTS SUMMARY — Step 17
# ============================================================

results_summary = f"""
=================================================================
CROSS-DATASET CANCER DETECTION — FULL RESULTS SUMMARY
Train: LC25000 (all 5 classes) | Test: Colorectal Histology MNIST
=================================================================

DATASET INFORMATION
-------------------
LC25000 (Training Dataset)
  Total images     : 25,000
  Classes          : 5 (colon_aca, colon_n, lung_aca, lung_n, lung_scc)
  Images per class : 5,000
  Image size       : 768 x 768 px (JPEG)
  Split            : 80% train (20,000) / 20% validation (5,000)

Colorectal Histology MNIST (Test Dataset)
  Total images     : 5,000 (8 classes)
  Classes used     : 2 (tumor, mucosa only)
  Images per class : 625
  Total evaluated  : 1,250
  Image size       : 150 x 150 px (TIF)

CLASS MAPPING APPLIED
---------------------
  Colorectal MNIST 'tumor'  → LC25000 'colon_aca' (idx=0) — malignant
  Colorectal MNIST 'mucosa' → LC25000 'colon_n'   (idx=1) — benign
  Excluded classes : stroma, lympho, debris, adipose, empty, complex
  Rationale        : Only biologically equivalent classes compared.
                     Lung classes trained but excluded from evaluation
                     as no colorectal equivalent exists.

MODEL INFORMATION
-----------------
  Architecture     : ResNet50 (pretrained on ImageNet)
  Custom head      : Dropout(0.4) → Linear(2048,256) → ReLU
                     → Dropout(0.3) → Linear(256,5)
  Epochs trained   : {checkpoint['epoch']}
  Optimizer        : Adam (lr=1e-4, weight_decay=1e-4)
  Scheduler        : ReduceLROnPlateau (factor=0.5, patience=3)
  Early stopping   : patience=5

=================================================================
PHASE 1 — LC25000 SAME-DATASET PERFORMANCE
=================================================================
  Best validation accuracy : {lc25000_val_acc:.4f} ({lc25000_val_acc*100:.2f}%)
  Note: Near-perfect score reflects dataset homogeneity from
        systematic augmentation, not genuine generalisation.

=================================================================
PHASE 2 — CROSS-DATASET EVALUATION (UNRESTRICTED)
All 5 LC25000 classes allowed in predictions
=================================================================
  Samples evaluated : {len(y_true)}
  Accuracy          : {accuracy:.4f} ({accuracy*100:.2f}%)
  Weighted F1 Score : {f1:.4f}
  AUC-ROC           : {auc_roc:.4f}
  Sensitivity       : {sensitivity:.4f}
  Specificity       : {specificity:.4f}

  Confusion Matrix:
    TP (cancer correctly identified) : {TP}
    FN (cancer missed)               : {FN}
    FP (benign wrongly flagged)      : {FP}
    TN (benign correctly passed)     : {TN}

  Prediction Distribution (all 5 classes):
    colon_aca : {list(all_preds.tolist()).count(LC25000_CLASS_TO_IDX['colon_aca'])}
    colon_n   : {list(all_preds.tolist()).count(LC25000_CLASS_TO_IDX['colon_n'])}
    lung_aca  : {list(all_preds.tolist()).count(LC25000_CLASS_TO_IDX['lung_aca'])}
    lung_n    : {list(all_preds.tolist()).count(LC25000_CLASS_TO_IDX['lung_n'])}
    lung_scc  : {list(all_preds.tolist()).count(LC25000_CLASS_TO_IDX['lung_scc'])}

  Key Finding: {list(all_preds.tolist()).count(LC25000_CLASS_TO_IDX['lung_scc'])} out of {len(all_preds)}
  predictions ({list(all_preds.tolist()).count(LC25000_CLASS_TO_IDX['lung_scc'])/len(all_preds)*100:.1f}%)
  were assigned to lung_scc despite all test images being
  colorectal tissue — evidence of severe texture bias under
  domain shift.

=================================================================
PHASE 3 — CROSS-DATASET EVALUATION (RESTRICTED)
Lung class probabilities zeroed — model forced to choose
between colon_aca and colon_n only
=================================================================
  Samples evaluated : {len(all_labels_filtered)}
  Accuracy          : {accuracy_f:.4f} ({accuracy_f*100:.2f}%)
  Weighted F1 Score : {f1_f:.4f}
  AUC-ROC           : {auc_f:.4f}
  Sensitivity       : {sensitivity_r:.4f}  ← cancer detection rate
  Specificity       : {specificity_r:.4f}  ← benign identification rate

  Confusion Matrix:
    TP (cancer correctly identified) : {TP_r}
    FN (cancer missed — critical)    : {FN_r}
    FP (benign wrongly flagged)      : {FP_r}
    TN (benign correctly passed)     : {TN_r}

  Prediction Distribution (restricted):
    colon_aca : {list(all_preds_filtered.tolist()).count(COLON_INDICES[0])}
    colon_n   : {list(all_preds_filtered.tolist()).count(COLON_INDICES[1])}

=================================================================
DOMAIN SHIFT ANALYSIS
=================================================================
  LC25000 accuracy        : {lc25000_val_acc*100:.2f}%
  Cross-dataset accuracy  : {accuracy*100:.2f}% (unrestricted)
  Restricted accuracy     : {accuracy_f*100:.2f}%
  Domain shift drop       : {(lc25000_val_acc - accuracy)*100:.2f}%

  Interpretation:
  The {(lc25000_val_acc - accuracy)*100:.2f}% accuracy drop confirms the original
  LC25000 results were inflated by dataset-specific memorisation.
  The model learned texture patterns tied to LC25000's 768x768
  JPEG format rather than genuine cancer biomarkers.

  Under restricted evaluation, sensitivity reaches {sensitivity_r*100:.1f}%
  but specificity remains low at {specificity_r*100:.1f}%, indicating
  partial transfer of cancer-positive features but failure to
  generalise benign tissue representations across domain shift.

  Resolution gap (768px → 150px) and format difference
  (JPEG → TIF) are identified as additional confounding
  factors contributing to domain shift beyond biological
  tissue differences.

=================================================================
CLINICAL IMPLICATION
=================================================================
  A model with {sensitivity_r*100:.1f}% sensitivity and {specificity_r*100:.1f}% specificity
  on unseen data would produce excessive false positives in a
  real screening setting, making it unsuitable for clinical
  deployment despite near-perfect same-dataset performance.
  This finding demonstrates the necessity of cross-dataset
  evaluation as a minimum standard for medical AI validation.

=================================================================
OUTPUT FILES SAVED
=================================================================
  training_history.png
  class_distributions.png
  confusion_matrix_cross_dataset.png
  roc_curve_cross_dataset.png
  performance_comparison.png
  prediction_distribution_analysis.png
  restricted_prediction_distribution.png
  restricted_confusion_matrix.png
  restricted_roc_curve.png
  full_metrics_comparison.png
  results_summary.txt
  restricted_results_summary.txt
=================================================================
"""

# Save to file
with open(RESULTS_DIR / 'full_results_summary.txt', 'w') as f:
    f.write(results_summary)

# Ensure the Google Drive directory exists
DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/cancer_detection_results')
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Also save to Google Drive
shutil.copy(
    RESULTS_DIR / 'full_results_summary.txt',
    DRIVE_SAVE_DIR / 'full_results_summary.txt'
)

print(results_summary)
print(f'Full results summary saved to:')
print(f'  {RESULTS_DIR}/full_results_summary.txt')
print(f'  Google Drive/cancer_detection_results/full_results_summary.txt')

## Step 18 — Save Model to Google Drive



In [ ]:
# Save model and results to Google Drive
DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/cancer_detection_results')
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy model
shutil.copy(BEST_MODEL_PATH, DRIVE_SAVE_DIR / 'best_lc25000_model.pth')

# Copy all results
for f in RESULTS_DIR.glob('*'):
    shutil.copy(f, DRIVE_SAVE_DIR / f.name)

print(f'Model and results saved to Google Drive: {DRIVE_SAVE_DIR}')
print('Files saved:')
for f in sorted(DRIVE_SAVE_DIR.glob('*')):
    print(f'  {f.name}')

## Step 19: Additional Checks

## 1.Grad_CAM

In [ ]:
# Install grad-cam library
!pip install grad-cam --quiet

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def show_gradcam(model, class_paths, transform, lc25000_class_to_idx,
                 n_samples=4, target_class='colon_aca'):

    target_idx = lc25000_class_to_idx[target_class]
    print(f'Target class: {target_class} (index={target_idx})')

    # Temporarily unfreeze weights so gradients can flow for Grad-CAM
    for param in model.parameters():
        param.requires_grad = True

    model.eval()
    target_layers = [model.layer4[-1]]
    cam = GradCAM(model=model, target_layers=target_layers)

    # Load sample images
    source_cls  = 'tumor' if 'aca' in target_class else 'mucosa'
    source_path = class_paths.get(source_cls)
    sample_imgs = (list(source_path.glob('*.tif')) +
                   list(source_path.glob('*.png')) +
                   list(source_path.glob('*.jpg')))[:n_samples]

    if not sample_imgs:
        print(f'No images found in {source_path}')
        return

    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples * 3, 6))
    fig.suptitle(f'Grad-CAM — What the model focuses on\n'
                 f'Source: Colorectal MNIST "{source_cls}" | '
                 f'Target class: "{target_class}"',
                 fontweight='bold')

    for i, img_path in enumerate(sample_imgs):
        img_pil     = Image.open(img_path).convert('RGB')
        img_resized = img_pil.resize((224, 224))
        img_np      = np.array(img_resized) / 255.0
        img_tensor  = transform(img_pil).unsqueeze(0).to(device)

        targets       = [ClassifierOutputTarget(target_idx)]
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)
        cam_image     = show_cam_on_image(
            img_np.astype(np.float32),
            grayscale_cam[0],
            use_rgb=True
        )

        axes[0][i].imshow(img_resized)
        axes[0][i].axis('off')
        axes[0][i].set_title(f'Original\n({source_cls})', fontsize=9)

        axes[1][i].imshow(cam_image)
        axes[1][i].axis('off')
        axes[1][i].set_title('Grad-CAM heatmap', fontsize=9)

    plt.tight_layout()
    save_path = RESULTS_DIR / f'gradcam_{target_class}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Grad-CAM saved: {save_path}')

    # Refreeze weights after Grad-CAM is done
    for param in model.parameters():
        param.requires_grad = False
    print('Weights refrozen after Grad-CAM.')


# Run for both colon classes
show_gradcam(model, COLORECTAL_CLASS_PATHS, val_test_transforms,
             LC25000_CLASS_TO_IDX, target_class='colon_aca')

show_gradcam(model, COLORECTAL_CLASS_PATHS, val_test_transforms,
             LC25000_CLASS_TO_IDX, target_class='colon_n')

## 2. Confidence Score Anlaysis

In [ ]:
# ---- Confidence Score Analysis ----
correct_mask   = (all_preds_filtered == all_labels_filtered)
incorrect_mask = ~correct_mask

# Get max probability (confidence) for each prediction
max_probs = np.max(all_probs, axis=1)

correct_conf   = max_probs[correct_mask]
incorrect_conf = max_probs[incorrect_mask]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Confidence Analysis — Cross-Dataset Evaluation',
             fontweight='bold')

# Histogram of confidence for correct vs incorrect
axes[0].hist(correct_conf, bins=20, alpha=0.7, color='#2ecc71',
             label=f'Correct (n={len(correct_conf)})', density=True)
axes[0].hist(incorrect_conf, bins=20, alpha=0.7, color='#C44E52',
             label=f'Incorrect (n={len(incorrect_conf)})', density=True)
axes[0].set_title('Confidence Distribution\nCorrect vs Incorrect', fontweight='bold')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Confidence by true class
cancer_mask = (all_labels_filtered == LC25000_CLASS_TO_IDX['colon_aca'])
benign_mask = (all_labels_filtered == LC25000_CLASS_TO_IDX['colon_n'])
cancer_conf = max_probs[cancer_mask]
benign_conf = max_probs[benign_mask]

axes[1].boxplot([cancer_conf, benign_conf],
                labels=['Cancer\n(tumor)', 'Benign\n(mucosa)'],
                patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.6))
axes[1].set_title('Confidence by True Class', fontweight='bold')
axes[1].set_ylabel('Confidence Score')
axes[1].grid(alpha=0.3)

# Overall confidence distribution
axes[2].hist(max_probs, bins=20, color='#4C72B0', alpha=0.8, edgecolor='white')
axes[2].axvline(x=0.5, color='red', linestyle='--', linewidth=1.5,
                label='0.5 threshold')
axes[2].axvline(x=np.mean(max_probs), color='orange', linestyle='--',
                linewidth=1.5, label=f'Mean={np.mean(max_probs):.3f}')
axes[2].set_title('Overall Confidence Distribution', fontweight='bold')
axes[2].set_xlabel('Confidence Score')
axes[2].set_ylabel('Count')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confidence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Confidence Analysis:')
print(f'  Mean confidence (correct)  : {np.mean(correct_conf):.4f}')
print(f'  Mean confidence (incorrect): {np.mean(incorrect_conf):.4f}')
print(f'  Mean confidence (cancer)   : {np.mean(cancer_conf):.4f}')
print(f'  Mean confidence (benign)   : {np.mean(benign_conf):.4f}')
print(f'  Overall mean confidence    : {np.mean(max_probs):.4f}')

## 3. Calibration Analysis

In [ ]:
# ---- Calibration Analysis ----
# A well calibrated model's confidence should match its actual accuracy
# e.g. when confidence is 80%, it should be correct ~80% of the time

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Calibration Analysis\n'
             '(Does confidence match actual accuracy?)',
             fontweight='bold')

# Bin predictions by confidence level
n_bins    = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_accs  = []
bin_confs = []
bin_counts = []

for i in range(n_bins):
    low, high = bin_edges[i], bin_edges[i+1]
    mask = (max_probs >= low) & (max_probs < high)
    if mask.sum() > 0:
        bin_acc  = (all_preds_filtered[mask] ==
                    all_labels_filtered[mask]).mean()
        bin_conf = max_probs[mask].mean()
        bin_accs.append(bin_acc)
        bin_confs.append(bin_conf)
        bin_counts.append(mask.sum())
    else:
        bin_accs.append(0)
        bin_confs.append((low + high) / 2)
        bin_counts.append(0)

# Reliability diagram
axes[0].plot([0,1],[0,1], 'k--', linewidth=1.5,
             label='Perfect calibration')
axes[0].bar(bin_confs, bin_accs, width=0.08, alpha=0.7,
            color='#4C72B0', label='Model calibration',
            edgecolor='white')
axes[0].set_xlabel('Mean Confidence', fontweight='bold')
axes[0].set_ylabel('Actual Accuracy', fontweight='bold')
axes[0].set_title('Reliability Diagram\n'
                  '(Bars above diagonal = overconfident)',
                  fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

# Confidence vs accuracy per bin as line
axes[1].plot(bin_confs, bin_accs, 'o-', color='#4C72B0',
             linewidth=2, markersize=8, label='Model')
axes[1].plot([0,1],[0,1], 'k--', linewidth=1.5,
             label='Perfect calibration')
axes[1].fill_between(bin_confs, bin_confs, bin_accs,
                     alpha=0.2, color='#C44E52',
                     label='Calibration gap')
axes[1].set_xlabel('Confidence', fontweight='bold')
axes[1].set_ylabel('Accuracy', fontweight='bold')
axes[1].set_title('Calibration Gap\n'
                  '(Red area = overconfidence region)',
                  fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'calibration_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Expected Calibration Error (ECE)
ece = sum(
    (counts / sum(bin_counts)) * abs(acc - conf)
    for counts, acc, conf in zip(bin_counts, bin_accs, bin_confs)
    if counts > 0
)
print(f'Expected Calibration Error (ECE): {ece:.4f}')
print()
print('Interpretation:')
print(f'  ECE of {ece:.4f} means on average the model confidence')
print(f'  is {ece*100:.1f}% away from its actual accuracy.')
print()
print('  A perfectly calibrated model has ECE = 0.0')
print('  ECE > 0.1 indicates significant overconfidence')
if ece > 0.1:
    print('  Your model IS overconfident on unseen data —')
    print('  a known risk factor for unreliable clinical AI.')
else:
    print('  Your model is reasonably calibrated on this metric.')

## 3. Visualise Misclassified Samples

In [ ]:
# ---- Visualise Misclassified Samples ----
def show_misclassified(dataset, labels, preds, lc25000_idx_to_class,
                       n_samples=8):
    misclassified_idx = np.where(labels != preds)[0]
    print(f'Total misclassified: {len(misclassified_idx)} / {len(labels)}')

    sample_idx = misclassified_idx[:n_samples]
    fig, axes  = plt.subplots(2, n_samples//2, figsize=(n_samples * 2, 6))
    axes       = axes.flatten()
    fig.suptitle('Misclassified Samples — Cross-Dataset Evaluation',
                 fontweight='bold')

    for plot_i, data_i in enumerate(sample_idx):
        path, true_label, cr_class = dataset.samples[data_i]
        pred_label = preds[data_i]
        img = Image.open(path).convert('RGB').resize((224, 224))

        true_name = lc25000_idx_to_class[true_label]
        pred_name = lc25000_idx_to_class[pred_label]

        axes[plot_i].imshow(img)
        axes[plot_i].axis('off')
        axes[plot_i].set_title(
            f'True: {true_name}\nPred: {pred_name}',
            fontsize=8, color='#C44E52', fontweight='bold'
        )

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'misclassified_samples.png',
                dpi=150, bbox_inches='tight')
    plt.show()

show_misclassified(
    colorectal_test_dataset,
    all_labels_filtered,
    all_preds_filtered,
    LC25000_IDX_TO_CLASS
)

In [ ]:
# ---- Per-Class Probability Distribution ----
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Predicted Probability Distribution Per LC25000 Class\n'
             '(on Colorectal MNIST images — all should be near 0 except colon classes)',
             fontweight='bold')

for i, (idx, cls_name) in enumerate(sorted(LC25000_IDX_TO_CLASS.items())):
    probs_for_class = all_probs[:, idx]
    color = '#4C72B0' if 'colon' in cls_name else '#C44E52'
    axes[i].hist(probs_for_class, bins=20, color=color,
                 alpha=0.8, edgecolor='white')
    axes[i].set_title(cls_name, fontweight='bold')
    axes[i].set_xlabel('Probability')
    axes[i].set_ylabel('Count' if i == 0 else '')
    axes[i].axvline(x=np.mean(probs_for_class), color='orange',
                    linestyle='--', linewidth=1.5,
                    label=f'Mean={np.mean(probs_for_class):.3f}')
    axes[i].legend(fontsize=8)
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_class_probability_dist.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Training Convergence Summary ----
history_df = pd.DataFrame({
    'Epoch'     : list(range(1, len(history['train_loss']) + 1)),
    'Train Loss': history['train_loss'],
    'Val Loss'  : history['val_loss'],
    'Train Acc' : [f'{a*100:.2f}%' for a in history['train_acc']],
    'Val Acc'   : [f'{a*100:.2f}%' for a in history['val_acc']]
})

print('Training History:')
print(history_df.to_string(index=False))
history_df.to_csv(RESULTS_DIR / 'training_history.csv', index=False)

# Find key convergence points
val_accs = history['val_acc']
best_epoch = np.argmax(val_accs) + 1
print(f'\nBest epoch          : {best_epoch}')
print(f'Best val accuracy   : {max(val_accs)*100:.2f}%')
print(f'Final train accuracy: {history["train_acc"][-1]*100:.2f}%')
print(f'Final val accuracy  : {history["val_acc"][-1]*100:.2f}%')
overfitting_gap = history['train_acc'][-1] - history['val_acc'][-1]
print(f'Overfitting gap     : {overfitting_gap*100:.2f}%')
if overfitting_gap > 0.05:
    print('  WARNING: Possible overfitting detected')
else:
    print('  Model generalised well on LC25000 validation set')

In [ ]:
# ---- Correct vs Misclassified Side by Side ----
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Correct vs Misclassified — Cancer Class (tumor → colon_aca)',
             fontweight='bold')

cancer_idx     = LC25000_CLASS_TO_IDX['colon_aca']
correct_idx    = np.where((all_labels_filtered == cancer_idx) &
                           (all_preds_filtered == cancer_idx))[0][:4]
incorrect_idx  = np.where((all_labels_filtered == cancer_idx) &
                           (all_preds_filtered != cancer_idx))[0][:4]

for j, idx in enumerate(correct_idx):
    path = colorectal_test_dataset.samples[idx][0]
    img  = Image.open(path).convert('RGB').resize((224, 224))
    axes[0][j].imshow(img)
    axes[0][j].axis('off')
    axes[0][j].set_title('✓ Correctly\nidentified', fontsize=9,
                          color='#2ecc71', fontweight='bold')

for j, idx in enumerate(incorrect_idx):
    path      = colorectal_test_dataset.samples[idx][0]
    pred_name = LC25000_IDX_TO_CLASS[all_preds_filtered[idx]]
    img       = Image.open(path).convert('RGB').resize((224, 224))
    axes[1][j].imshow(img)
    axes[1][j].axis('off')
    axes[1][j].set_title(f'✗ Predicted as\n{pred_name}', fontsize=9,
                          color='#C44E52', fontweight='bold')

axes[0][0].set_ylabel('Correct', fontsize=11, fontweight='bold')
axes[1][0].set_ylabel('Incorrect', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'correct_vs_misclassified.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import shutil
from pathlib import Path

DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/cancer_detection_results')
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

for f in RESULTS_DIR.glob('*'):
    shutil.copy(f, DRIVE_SAVE_DIR / f.name)

print('All results backed up to Google Drive.')
print('Files saved:')
for f in sorted(DRIVE_SAVE_DIR.glob('*')):
    print(f'  {f.name}')

---
## Summary of What This Notebook Does

| Step | What happens | Why |
|------|-------------|-----|
| 1–2 | Install dependencies, mount Drive | Setup |
| 3–4 | Download both datasets, verify structure | Data acquisition |
| 5 | Visualise class distributions and samples | Understand your data |
| 6 | Define train vs test transforms | No augmentation leaks into test |
| 7 | Build LC25000 DataLoaders (80/20 split) | All 5 classes for training |
| 8 | Build ResNet50 model | Pretrained backbone |
| 9–10 | Train with early stopping, plot history | Train only on LC25000 |
| 11 | Class mapping — build Colorectal MNIST loader | Only tumor + mucosa |
| 12 | Load best model, freeze weights, run inference | No retraining |
| 13–15 | Accuracy, F1, AUC-ROC, confusion matrix, ROC curve | Full evaluation |
| 16 | Compare LC25000 vs cross-dataset performance | Core finding |
| 17–18 | Save results and model to Drive | Persistence |



---
## Updated Summary of What This Notebook Does

| Step | What happens | Why |
|------|-------------|-----|
| 1–2 | Install dependencies, mount Drive, configure Kaggle API | Setup |
| 3–4 | Download both datasets, auto-detect folder paths, build unified LC25000 structure | Data acquisition + path fixing |
| 5 | Visualise class distributions and sample images using resolved paths | Understand your data |
| 6 | Define train vs test transforms (resize, augment, normalize) | No augmentation leaks into test |
| 7 | Build unified LC25000 DataLoaders via symlinks (80/20 split) | All 5 classes for training |
| 8 | Build ResNet50 model with custom classification head | Pretrained ImageNet backbone |
| 9–10 | Train with early stopping and LR scheduler, plot history | Train only on LC25000 |
| 11 | Class mapping — build Colorectal MNIST loader using COLORECTAL_CLASS_PATHS | Only tumor + mucosa (2 mapped classes) |
| 12 | Load best model, freeze all weights, run unrestricted inference | No retraining — pure generalization test |
| 13–15 | Accuracy, F1, AUC-ROC, confusion matrix, ROC curve (unrestricted) | Full unrestricted evaluation |
| 16 | Compare LC25000 vs cross-dataset performance — domain shift chart | Core finding |
| 17 | Restricted evaluation — zero out lung predictions, rerun all metrics | Fairer colon-only evaluation |
| 18 | Full visualisation of restricted results (4 plots + 3-way comparison) | Restricted vs unrestricted comparison |
| 19 | Updated full results summary saved to Drive | Complete documentation |

---

## Key Findings Summary

| Finding | Value | Interpretation |
|---------|-------|----------------|
| LC25000 validation accuracy | 99.98% | Inflated — dataset memorisation |
| Cross-dataset accuracy (unrestricted) | 12.08% | Severe domain shift |
| Cross-dataset accuracy (restricted) | see output | Fairer comparison |
| lung_scc predictions on colorectal images | 833/1250 (66.6%) | Texture bias under domain shift |
| Sensitivity (restricted) | 99.0% | Cancer features partially transfer |
| Specificity (restricted) | 13.4% | Benign features do not transfer |
| Domain shift drop | 87.90% | Core research finding |

